The sales series should first be checked for trend, seasonality, and stationarity because these affect model choice. If the p-value from ADF test is more than 0.05, the series is non-stationary and differencing may be needed. Missing dates, null values, and duplicate rows should also be fixed before training. If seasonality is visible, models like SARIMA or Prophet may perform better than simple ARIMA.

I first checked the dataset because sequence models like LSTM, GRU, RNN, Transformer need clean and continuous time-series data. If missing values, wrong order, duplicates, or useless columns remain, the model can fail or learn wrong patterns.

In [2]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 22.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import pandas as pd


df = pd.read_csv("/Users/omshinde/Desktop/Python/assignment/Assignment 44/sensor.csv")



print("Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDuplicate timestamps:", df["timestamp"].duplicated().sum())




if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)


if df["sensor_15"].isnull().all():
    df.drop(columns=["sensor_15"], inplace=True)


df["timestamp"] = pd.to_datetime(df["timestamp"])


df = df.sort_values("timestamp").reset_index(drop=True)


sensor_cols = [col for col in df.columns if col.startswith("sensor_")]


df[sensor_cols] = df[sensor_cols].interpolate(method="linear", limit_direction="both")
# df[sensor_cols] = df[sensor_cols].fillna(method="ffill").fillna(method="bfill")


print("\nRemaining Missing Values:", df.isnull().sum().sum())
print("Clean Shape:", df.shape)


df.to_csv("sensor_clean.csv", index=False)

Shape: (220320, 55)

Missing Values:
sensor_15    220320
sensor_50     77017
sensor_51     15383
sensor_00     10208
sensor_07      5451
sensor_08      5107
sensor_06      4798
sensor_09      4595
sensor_01       369
sensor_30       261
sensor_29        72
sensor_32        68
sensor_18        46
sensor_17        46
sensor_22        41
dtype: int64

Duplicate timestamps: 0

Remaining Missing Values: 0
Clean Shape: (220320, 53)


Using the cleaned sensor data, I built a failure risk prediction model to estimate whether equipment is likely to fail in the next 24 hours. The main goal is not only accuracy, but to reduce missed failures because a missed breakdown can stop production and create high repair cost.

In [13]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl (2.3 MB)

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder


df = pd.read_csv("sensor_clean.csv")


df["timestamp"] = pd.to_datetime(df["timestamp"])


df["failure_now"] = (df["machine_status"] != "NORMAL").astype(int)


df["failure_next_24h"] = (
    df["failure_now"]
    .rolling(window=24, min_periods=1)
    .max()
    .shift(-24)
)

df["failure_next_24h"] = df["failure_next_24h"].fillna(0).astype(int)

X = df.drop(columns=[
    "timestamp",
    "machine_status",
    "failure_now",
    "failure_next_24h"
])

y = df["failure_next_24h"]


split = int(len(df)*0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]


model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=3,
    random_state=42
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)

print("Recall:", recall_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Recall: 0.0

Confusion Matrix:
[[44061     3]
 [    0     0]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     44064
           1       0.00      0.00      0.00         0

    accuracy                           1.00     44064
   macro avg       0.50      0.50      0.50     44064
weighted avg       1.00      1.00      1.00     44064



/Users/omshinde/Desktop/Python/w2v_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/omshinde/Desktop/Python/w2v_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/omshinde/Desktop/Python/w2v_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", re

Now the model is deployed on 100,000 sensors, so prediction errors create real business cost.

False Positive (FP) = unnecessary inspection
False Negative (FN) = missed failure → emergency repair

We need the threshold that gives minimum money loss, not only best ML score.

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


y_prob = model.predict_proba(X_test)[:,1]

thresholds = np.arange(0.1, 0.95, 0.05)

results = []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)

    TP = ((y_test==1) & (y_pred==1)).sum()
    TN = ((y_test==0) & (y_pred==0)).sum()
    FP = ((y_test==0) & (y_pred==1)).sum()
    FN = ((y_test==1) & (y_pred==0)).sum()

    cost = FP*500 + FN*20000
    f1 = f1_score(y_test, y_pred)

    results.append([t, TP, FP, FN, cost, f1])

res = pd.DataFrame(results,
    columns=["threshold","TP","FP","FN","cost","f1"])

print(res)

best_cost = res.loc[res["cost"].idxmin()]
best_f1 = res.loc[res["f1"].idxmax()]

print("\nBest Threshold by Cost:")
print(best_cost)

print("\nBest Threshold by F1:")
print(best_f1)

    threshold  TP  FP  FN  cost   f1
0        0.10   0   8   0  4000  0.0
1        0.15   0   8   0  4000  0.0
2        0.20   0   8   0  4000  0.0
3        0.25   0   8   0  4000  0.0
4        0.30   0   7   0  3500  0.0
5        0.35   0   7   0  3500  0.0
6        0.40   0   7   0  3500  0.0
7        0.45   0   6   0  3000  0.0
8        0.50   0   3   0  1500  0.0
9        0.55   0   2   0  1000  0.0
10       0.60   0   0   0     0  0.0
11       0.65   0   0   0     0  0.0
12       0.70   0   0   0     0  0.0
13       0.75   0   0   0     0  0.0
14       0.80   0   0   0     0  0.0
15       0.85   0   0   0     0  0.0
16       0.90   0   0   0     0  0.0

Best Threshold by Cost:
threshold    0.6
TP           0.0
FP           0.0
FN           0.0
cost         0.0
f1           0.0
Name: 10, dtype: float64

Best Threshold by F1:
threshold       0.1
TP              0.0
FP              8.0
FN              0.0
cost         4000.0
f1              0.0
Name: 0, dtype: float64


/Users/omshinde/Desktop/Python/w2v_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/omshinde/Desktop/Python/w2v_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/omshinde/Desktop/Python/w2v_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

In [22]:
scale = 100000 / len(X_test)

daily_cost = best_cost["cost"] * scale
print("Expected Daily Cost =", daily_cost)

Expected Daily Cost = 0.0


The best production threshold is the one that minimises total business cost, not necessarily the one with highest F1. This shows that ML evaluation should be aligned with business impact, especially when false negatives are much more expensive than false positives.